# SPR addresses → cadastre id · walkthrough

Run this notebook **top to bottom, one cell at a time**. Each step saves its tables in `work/mapping.duckdb`, and the cells after it show what that step produced, so you can stop and check before going on.

**Rule:** an SPR row gets a `cadastre_id` only when exactly one cadastre address fits and the place, street, house and apartment all agree. Anything weaker becomes a suggestion or an issue with a named owner (CADASTRE / SPR / REVIEW).

Tools for investigating, usable after any step:

| function | what it shows |
|---|---|
| `explain(spr_id)` | everything the pipeline did with one SPR row |
| `search_cadastre('բաղրամյան', 'Արաբկիր')` | cadastre streets containing a text, optionally in one place |
| `streets_in('Արաբկիր')` | every cadastre street of a settlement / Yerevan district |
| `houses_on(street_uid)` | every cadastre address on one street |
| `sample('CAD_HOUSE_NOT_FOUND', 20)` | random rows of one issue category |
| `q("SELECT ...")` | any SQL on the working tables |

Input files go in `data/input/`; paths are in `config.py`.

In [4]:
import os, sys
sys.path.insert(0, os.getcwd())   # open this notebook from the project folder (the one with config.py)
assert os.path.exists('config.py'), 'Open the notebook from the project folder'

from steps import s01_load, s02_cadastre, s03_prepare_spr, s04_match_streets, s05_match_addresses, s06_export, s07_checks
from mapping.explore import q, explain, search_cadastre, streets_in, houses_on, sample

## Step 1 · Load the two files, untouched

- `raw_cadastre`, `raw_spr`: the files exactly as received.
- `spr`: `raw_spr` plus **`spr_id`**, an identity key 1..N in file order. Every output carries it.
- `match_type` / `cad_candidate_ids` (someone else's mapping) stay in the table but are never read.

In [2]:
s01_load.run()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[01_load] cadastre rows: 1,048,575   spr rows: 614,083
[01_load] WARNING: cadastre has exactly the Excel row limit; the export is probably truncated


In [3]:
q("""SELECT 'cadastre' AS file, count(*) AS rows FROM cadastre
     UNION ALL SELECT 'spr', count(*) FROM spr""")

,file,rows
0,cadastre,1048575
1,spr,614083


In [4]:
q('SELECT * FROM spr ORDER BY spr_id LIMIT 5')

,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,1,AMBIGUOUS,"1000006,1252080,999599",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ՔԱՆԱՔԵՌ 12 Փ.,15,Տ,NaN
1,2,AMBIGUOUS,"1000006,1252080,999599",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.,15,Տ,NaN
2,3,AMBIGUOUS,"10000,1228713,1470303",ՇԻՐԱԿ,ԱՇՈՑՔ,ԱՇՈՑՔ,3 ԹՂՄ. 4 Փ.,3,Շ,3
3,4,AMBIGUOUS,"1000014,1251663",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ԶԱՔԱՐԻԱ ՔԱՆԱՔԵՌՑՈՒ Փ.,151/1,Շ,38
4,5,AMBIGUOUS,"1000019,999195",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ԶԱՔԱՐԻԱ ՔԱՆԱՔԵՌՑՈՒ Փ.,151/1,Շ,11


**Check:** a cadastre file with exactly **1,048,575** rows hit Excel's sheet limit, so the export is probably cut off. Houses missing in later steps may be missing only from this file.

## Step 2 · Cadastre keys and cadastre data issues

- `cadastre_keys`: one row per cadastre id with cleaned geography, a `street_uid` and cleaned house / building numbers.
- `cadastre_streets`: one row per distinct street per settlement.
- `cadastre_issues`: problems in the cadastre itself, found without looking at SPR.

In [5]:
s02_cadastre.run()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[02_cadastre] cadastre streets: 36,446
[02_cadastre]    CAD_DUPLICATE_ADDRESS               90,073 ids
[02_cadastre]    CAD_SAME_ADDRESS_MULTI_OBJECT       70,634 ids
[02_cadastre]    CAD_STREET_NAME_MISSING              7,228 ids
[02_cadastre]    CAD_NONSTANDARD_HOUSE_NUMBER         6,601 ids
[02_cadastre]    CAD_LEADING_TRAILING_SPACES          3,679 ids
[02_cadastre]    CAD_STREET_SPELLING_VARIANTS            37 ids


In [6]:
q('SELECT * FROM cadastre_streets USING SAMPLE 10 ROWS (reservoir, 1)')

,street_uid,region_k,community_k,settlement_k,street_key,variant_key,street_name,street_type,sub_street_name,sub_street_type,region,community,settlement,n_addresses,is_spelling_variant
0,148,արագածոտն,աշտարակ,ագարակ,12|փողոց|2|փակուղի,12|փողոց|2|փակուղի,12-րդ,փողոց,2-րդ,փակուղի,Արագածոտն,Աշտարակ,Ագարակ,2,False
1,6848,արարատ,մասիս,նոր կյուրին,11|փողոց||,11|փողոց||,11-րդ,փողոց,NaN,NaN,Արարատ,Մասիս,Նոր Կյուրին,9,False
2,9608,արմավիր,մեծամոր,ակնալիճ,հոկտեմբերյան|փողոց||,հոկտեմբերյան|փողոց||,Հոկտեմբերյան,փողոց,NaN,NaN,Արմավիր,Մեծամոր,Ակնալիճ,59,False
3,17793,լոռի,ալավերդի,ախթալա,ս սպանդարյան|փողոց||,սպանդարյան|փողոց||,Ս. Սպանդարյան,փողոց,NaN,NaN,Լոռի,Ալավերդի,Ախթալա,44,False
4,20112,լոռի,ստեփանավան,ստեփանավան,շաղարդա|թաղամաս||,շաղարդա|թաղամաս||,Շաղարդա,թաղամաս,NaN,NaN,Լոռի,Ստեփանավան,Ստեփանավան,3,False
5,20606,լոռի,վանաձոր,վանաձոր,սանկտ պետերբուրգի|փողոց||,սանկտ պետերբուրգ|փողոց||,Սանկտ-Պետերբուրգի,փողոց,NaN,NaN,Լոռի,Վանաձոր,Վանաձոր,146,False
6,25998,կոտայք,ջրվեժ,ձորաղբյուր,ազատության|փողոց|11|նրբանցք,ազատության|փողոց|11|նրբանցք,Ազատության,փողոց,11-րդ,նրբանցք,Կոտայք,Ջրվեժ,Ձորաղբյուր,7,False
7,26893,շիրակ,ախուրյան,ղարիբջանյան,8|փողոց|1|նրբանցք,8|փողոց|1|նրբանցք,8-րդ,փողոց,1-ին,նրբանցք,Շիրակ,Ախուրյան,Ղարիբջանյան,3,False
8,32328,վայոց ձոր,եղեգիս,քարագլուխ,5|փողոց|1|նրբանցք,5|փողոց|1|նրբանցք,5-րդ,փողոց,1-ին,նրբանցք,Վայոց ձոր,Եղեգիս,Քարագլուխ,6,False
9,34230,տավուշ,դիլիջան,դիլիջան,այգեստան|փողոց||,այգեստան|փողոց||,Այգեստան,փողոց,NaN,NaN,Տավուշ,Դիլիջան,Դիլիջան,117,False


In [7]:
q('SELECT issue_code, count(*) AS cadastre_ids FROM cadastre_issues GROUP BY 1 ORDER BY 2 DESC')

,issue_code,cadastre_ids
0,CAD_DUPLICATE_ADDRESS,90073
1,CAD_SAME_ADDRESS_MULTI_OBJECT,70634
2,CAD_STREET_NAME_MISSING,7228
3,CAD_NONSTANDARD_HOUSE_NUMBER,6601
4,CAD_LEADING_TRAILING_SPACES,3679
5,CAD_STREET_SPELLING_VARIANTS,37


Examples: identical cadastre addresses with several ids.

In [8]:
q("""SELECT i.detail, c.region, c.settlement, c.street_name, c.street_type, c.house, c.house_type, c.building
     FROM cadastre_issues i JOIN cadastre c ON c.id = i.cadastre_id
     WHERE i.issue_code = 'CAD_DUPLICATE_ADDRESS' ORDER BY random() LIMIT 10""")

,detail,region,settlement,street_name,street_type,house,house_type,building
0,"same address and same type; ids: 265207,265208,265214",Արագածոտն,Ավան,1-ին,փողոց,1,հողամաս,NaN
1,"same address and same type; ids: 729826,729829",Երևան,Արաբկիր,Ա. Խաչատրյան,փողոց,3,շենք,145
2,"same address and same type; ids: 1117018,1117019,1117020,1117021,1...",Շիրակ,Ղազանչի,1-ին,փողոց,11,բնակելի տուն,NaN
3,"same address and same type; ids: 1207055,384881",Արագածոտն,Թալին,Ժ. Ավետիսյան,փողոց,48,բնակելի տուն,NaN
4,"same address and same type; ids: 1204739,254427",Արարատ,Այնթապ,20-րդ,փողոց,17,հողամաս,NaN
5,"same address and same type; ids: 1185388,554766",Լոռի,Ստեփանավան,Ֆ. Նանսենի,փողոց,40,NaN,NaN
6,"same address and same type; ids: 1212930,408688",Արմավիր,Մերձավան,2-րդ,փողոց,45,հողամաս,NaN
7,"same address and same type; ids: 1103395,209134",Տավուշ,Սարիգյուղ,7-րդ,փողոց,5,հողամաս,NaN
8,"same address and same type; ids: 1233985,673970",Վայոց ձոր,Եղեգնաձոր,Թամանցիների,փողոց,16,բնակելի տուն,NaN
9,"same address and same type; ids: 548863,549283",Լոռի,Տաշիր,Անի,զանգված,15,շենք,11


## Step 3 · Put SPR geography on the cadastre's hierarchy

Colleagues' corrections in their order: `geo_alias` → `community_alias` → Yerevan (district moves from community to settlement) → Gyumri → as is.
Each row also gets the colleagues' `data_completeness` (% of the 6 address parts that are filled).

In [2]:
from steps import s03_prepare_spr
s03_prepare_spr.run()

[03_prepare_spr] rows: 614,083
[03_prepare_spr]    geo_status ok                             515,774
[03_prepare_spr]    geo_status foreign_or_unknown_region       96,297
[03_prepare_spr]    geo_status region_missing                   1,683
[03_prepare_spr]    geo_status community_not_in_cadastre          279
[03_prepare_spr]    geo_status settlement_not_in_cadastre          50
[03_prepare_spr]    geo_rule   as_is                          409,690
[03_prepare_spr]    geo_rule   yerevan                        156,624
[03_prepare_spr]    geo_rule   gyumri                          39,753
[03_prepare_spr]    geo_rule   geo_alias                        7,843
[03_prepare_spr]    geo_rule   community_alias                    173


In [5]:
q('SELECT geo_status, geo_note, count(*) AS rows FROM spr_prepared GROUP BY ALL ORDER BY rows DESC')

,geo_status,geo_note,rows
0,ok,NaN,505823
1,foreign_or_unknown_region,NaN,96297
2,ok,district_missing,9108
3,region_missing,NaN,1683
4,ok,settlement_missing,809
5,community_not_in_cadastre,NaN,279
6,settlement_not_in_cadastre,NaN,50
7,ok,community_missing,23
8,ok,district_not_in_cadastre,11


In [6]:
q('SELECT geo_rule, count(*) AS rows FROM spr_prepared GROUP BY 1 ORDER BY 2 DESC')

,geo_rule,rows
0,as_is,409690
1,yerevan,156624
2,gyumri,39753
3,geo_alias,7843
4,community_alias,173


Corrections in action (one example per rule):

In [7]:
q("""SELECT DISTINCT ON (p.geo_rule) p.geo_rule, s.region, s.community, s.settlement,
            p.community_k AS community_used, p.settlement_k AS settlement_used
     FROM spr_prepared p JOIN spr s USING (spr_id)""")

,geo_rule,region,community,settlement,community_used,settlement_used
0,gyumri,ՇԻՐԱԿ,ԳՅՈՒՄՐԻ,NaN,գյումրի,գյումրի
1,community_alias,ԿՈՏԱՅՔ,ՄԵՂՐԱՁՈՐ ԲԱՆԱՎԱՆ,NaN,մեղրաձոր,
2,as_is,ԿՈՏԱՅՔ,ԱԲՈՎՅԱՆ,ԿԱԹՆԱՂԲՅՈՒՐ,աբովյան,կաթնաղբյուր
3,geo_alias,ԿՈՏԱՅՔ,ԵՂՎԱՐԴ,NaN,նաիրի,եղվարդ
4,yerevan,ԵՐԵՎԱՆ,13,NaN,երեվան,13


Places that do not exist in the cadastre (with a hint when the settlement is found under another community):

In [8]:
q("""SELECT s.region, s.community, s.settlement, p.geo_status, p.geo_hint, count(*) AS rows
     FROM spr_prepared p JOIN spr s USING (spr_id)
     WHERE p.geo_status IN ('community_not_in_cadastre', 'settlement_not_in_cadastre')
     GROUP BY ALL ORDER BY rows DESC LIMIT 20""")

,region,community,settlement,geo_status,geo_hint,rows
0,ԿՈՏԱՅՔ,ՄԵՂՐԱՁՈՐ ԲԱՆԱՎԱՆ,NaN,community_not_in_cadastre,is a settlement of community: ծաղկաձոր,42
1,ԱՐԱՐԱՏ,ՎԵԴՈՒ Գ/Գ ԲԱՆԱՎԱՆ,NaN,community_not_in_cadastre,NaN,25
2,ԱՐԱԳԱԾՈՏՆ,ԱՄՐԵ ԹԱԶԱ,NaN,community_not_in_cadastre,NaN,17
3,ՇԻՐԱԿ,ԳՅՈՒՄՐԻ,ԱՐՏԵՆԻ,settlement_not_in_cadastre,NaN,15
4,ՇԻՐԱԿ,ԳՅՈՒԼԼԻԲՈՒԼԱՂ,NaN,community_not_in_cadastre,NaN,14
5,ԱՐԱՐԱՏ,ԶԻՆԱՎԱՆ 5,NaN,community_not_in_cadastre,NaN,14
6,ՍՅՈՒՆԻՔ,ՈՒԽՏԱՎԱՅՐ,NaN,community_not_in_cadastre,NaN,13
7,ՍՅՈՒՆԻՔ,ԽԱԼԱՋ,NaN,community_not_in_cadastre,NaN,13
8,ՍՅՈՒՆԻՔ,ՆԺԴԵՀ,ՄՈՒՐԽՈՒԶ,community_not_in_cadastre,is a settlement of community: սիսիան,11
9,ՇԻՐԱԿ,ՋԱՋՈՒՌ ԿԱՅԱՐԱՆ,NaN,community_not_in_cadastre,NaN,7


In [9]:
q('SELECT data_completeness, count(*) AS rows FROM spr_prepared GROUP BY 1 ORDER BY 1 DESC')

,data_completeness,rows
0,100.00,113721
1,83.33,341747
2,66.67,92957
3,50.00,39067
4,33.33,24804
5,16.67,1787


## Step 4 · Find the cadastre street

For each search area (settlement first), each rule in order, each reading of the text; the **first hit decides**.

| # | rule | allows | confidence |
|---|---|---|---|
| 1 | exact | identical cleaned name | high |
| 2 | alias | colleagues' renames (Գյումրի, Կապան) | high |
| 3 | double_letter | notebook typo list, doubled letters | high |
| 4 | word_form | genitive endings (`ԳԱՌՆԻԻ` = `Գառնու`) | high |
| 5 | extra_words | SPR words ⊂ cadastre words (`ԱՆԴՐԱՆԻԿԻ` = `Զորավար Անդրանիկի`) | medium |
| 6 | initial_for_name | `Ք.` = `ՔԸՐՔ`, or one side omits the initial | medium |
| 7 | fuzzy | letters only, score ≥ 90 and 4 clear of the next street | medium |

Numbers (street number, block code, lane) are never fuzzy: they must be identical and in the same order.

Try the street parser on any text first:

In [10]:
from mapping.street_parser import parse_spr_street
for text in ['ՍՈՍԵՅԻ Փ. 1 ՆՐԲ.', 'Ա2 ԹՂՄ. 9 Փ.', 'ԴՐՈՅԻ 4 ՆՐԲ.', 'ՔԸՐՔ ՔԸՐՔՈՐՅԱՆ Փ.', 'ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.']:
    print(f'{text:22}', [(r, st.text(), leftover) for r, st, leftover, _ in parse_spr_street(text)])

ՍՈՍԵՅԻ Փ. 1 ՆՐԲ.       [('as_written', 'սոսեյի | փողոց | 1 | նրբանցք', 'none')]
Ա2 ԹՂՄ. 9 Փ.           [('neighbourhood_street', 'ա2 թաղամասի 9 | փողոց', 'none')]
ԴՐՈՅԻ 4 ՆՐԲ.           [('as_written', 'դրոյի 4 | նրբանցք', 'none'), ('lane_of_street', 'դրոյի | 4 | նրբանցք', 'none')]
ՔԸՐՔ ՔԸՐՔՈՐՅԱՆ Փ.      [('as_written', 'քըրք քըրքորյան | փողոց', 'none')]
ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.    []


In [11]:
s04_match_streets.run()

[04_match_streets] distinct (scope, street) pairs to match: 48,812
[04_match_streets]    20,000 done
[04_match_streets]    40,000 done
[04_match_streets] rows matched at street level: 515,774
[04_match_streets]    found                  391,235
[04_match_streets]    not_found               66,814
[04_match_streets]    type_mismatch           23,611
[04_match_streets]    suggested               16,275
[04_match_streets]    no_readings             14,440
[04_match_streets]    ambiguous                2,530
[04_match_streets]    no_street_in_spr           869


In [12]:
q('SELECT street_status, count(*) AS rows FROM spr_street_match GROUP BY 1 ORDER BY 2 DESC')

,street_status,rows
0,found,391235
1,not_found,66814
2,type_mismatch,23611
3,suggested,16275
4,no_readings,14440
5,ambiguous,2530
6,no_street_in_spr,869


In [13]:
q("""SELECT match_method, text_reading, type_check, count(*) AS rows FROM spr_street_match
     WHERE street_status = 'found' GROUP BY ALL ORDER BY rows DESC""")

,match_method,text_reading,type_check,rows
0,exact,as_written,same type,204691
1,initial_for_name,as_written,same type,93248
2,extra_words,as_written,same type,43800
3,word_form,as_written,same type,13046
4,exact,as_written,issue has no type,7052
5,exact,neighbourhood_street,same type,6805
6,exact,lane_of_street,issue has no type,4107
7,initial_for_name,lane_of_street,issue has no type,4086
8,fuzzy,as_written,same type,3200
9,word_form,without_house_numbers,issue has no type,3052


SPR street next to the cadastre street it was matched to — change `METHOD` to check any rule:

In [14]:
METHOD = 'initial_for_name'   # exact, alias, double_letter, word_form, extra_words, initial_for_name, fuzzy
q(f"""SELECT s.settlement, s.community, s.street AS spr_street, m.text_reading,
            cs.street_name, cs.street_type, cs.sub_street_name, cs.sub_street_type, m.similarity_score
     FROM spr_street_match m JOIN spr s USING (spr_id) JOIN cadastre_streets cs ON cs.street_uid = m.street_uid
     WHERE m.street_status = 'found' AND m.match_method = '{METHOD}'
     ORDER BY random() LIMIT 20""")

,settlement,community,spr_street,text_reading,street_name,street_type,sub_street_name,sub_street_type,similarity_score
0,ՆՈՅԵՄԲԵՐՅԱՆ,ՆՈՅԵՄԲԵՐՅԱՆ,ԱԲՈՎՅԱՆ Փ.,as_written,Խ. Աբովյան,փողոց,None,None,NaN
1,ՍՏԵՓԱՆԱՎԱՆ,ՍՏԵՓԱՆԱՎԱՆ,ՏԵՐՅԱՆ Փ.,as_written,Վ. Տերյան,փողոց,None,None,NaN
2,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ԹԵՎՈՍՅԱՆ Փ.,as_written,Ս. Թևոսյան,փողոց,None,None,NaN
3,ԱԳԱՐԱԿ,ՄԵՂՐԻ,ՉԱՐԵՆՑԻ Փ.,as_written,Ե. Չարենցի,փողոց,None,None,NaN
4,NaN,ԳՅՈՒՄՐԻ,ՌԱՍԿԱՏԼՅԱՆ Փ.,as_written,Հ. Ռասկատլյան,փողոց,None,None,NaN
5,NaN,ԴԱՎԹԱՇԵՆ,ՅՈՒՐԻ ԵՐԶՆԿՅԱՆ Փ.,as_written,Յ. Երզնկյան,փողոց,None,None,NaN
6,ԲՅՈՒՐԵՂԱՎԱՆ,ԲՅՈՒՐԵՂԱՎԱՆ,ՎԱԶԳԵՆ ՍԱՐԳՍՅԱՆ Փ.,as_written,Վ. Սարգսյան,փողոց,None,None,NaN
7,ԱՐՄԱՎԻՐ,ԱՐՄԱՎԻՐ,ՇԱՀՈՒՄՅԱՆ Փ.,as_written,Ս. Շահումյան,փողոց,None,None,NaN
8,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ԽՈՐԵՆԱՑՈՒ Փ.,as_written,Մ. Խորենացու,փողոց,None,None,NaN
9,ՆՈՐԱԿԵՐՏ,ՓԱՐԱՔԱՐ,ՏԵՐՅԱՆ Փ.,as_written,Վ. Տերյան,փողոց,None,None,NaN


Most frequent street texts that were **not found** (biggest impact first):

In [15]:
q("""SELECT s.region, s.community, s.settlement, s.street, m.parsed_street, count(*) AS rows
     FROM spr_street_match m JOIN spr s USING (spr_id) WHERE m.street_status = 'not_found'
     GROUP BY ALL ORDER BY rows DESC LIMIT 20""")

,region,community,settlement,street,parsed_street,rows
0,ՇԻՐԱԿ,ԱԽՈՒՐՅԱՆ,ԱԽՈՒՐՅԱՆ,ՇՐՋԿԵՆՏՐՈՆ,շրջկենտրոն,1152
1,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,ՋՐՎԵԺ ՄԱՅԱԿ,ջրվեժ մայակ,1131
2,ԵՐԵՎԱՆ,ԿԵՆՏՐՈՆ,NaN,ՊԱՊ ԹԱԳԱՎՈՐԻ Փ.,պապ թագավորի | փողոց,988
3,ԼՈՌԻ,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ՏԱՐՈՆ-4 ՄԵՔ.,տարոն 4 մեք,980
4,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,ՋՐՎԵԺ ԲԱՆԱՎԱՆ,ջրվեժ բանավան,897
5,ԵՐԵՎԱՆ,ԱՐԱԲԿԻՐ,NaN,ՔԱՆԱՔԵՌ ՀԷԿ ԲԱՆԱՎԱՆ,քանաքեռ հէկ բանավան,718
6,ԵՐԵՎԱՆ,ԱՎԱՆ,NaN,ԱՎԱՆ ԱՌԻՆՋ,ավան առինջ,705
7,ԵՐԵՎԱՆ,ՇԵՆԳԱՎԻԹ,NaN,Ն. ՇԵՆԳԱՎԻԹ 11 Փ.,ն շենգավիթ 11 | փողոց,649
8,ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,Մ. ԱՎԵՏԻՍՅԱՆ 4 Փ.,մ ավետիսյան 4 | փողոց,625
9,ԿՈՏԱՅՔ,ԱԲՈՎՅԱՆ,ԱԲՈՎՅԱՆ,ՉԱՓԻՉ,չափիչ,580


Name found but type differs (`Փ.` vs `պողոտա` …) — not mapped:

In [16]:
q("""SELECT s.settlement, s.community, s.street, m.parsed_street, cs.street_name, cs.street_type, count(*) AS rows
     FROM spr_street_match m JOIN spr s USING (spr_id)
     JOIN cadastre_streets cs ON cs.street_uid = TRY_CAST(split_part(m.candidate_uids, ',', 1) AS BIGINT)
     WHERE m.street_status = 'type_mismatch' GROUP BY ALL ORDER BY rows DESC LIMIT 15""")

,settlement,community,street,parsed_street,street_name,street_type,rows
0,NaN,ՆՈՐ ՆՈՐՔ,ՆՈՐ ՆՈՐՔ 7 ԶՆԳՎ.,նոր նորք 7 | զանգված,7-րդ,միկրոշրջան,2321
1,NaN,ՇԵՆԳԱՎԻԹ,ԱՐՏԱՇԻՍՅԱՆ Փ.,արտաշիսյան | փողոց,Արտաշեսյան,պողոտա,1850
2,NaN,ՆՈՐ ՆՈՐՔ,ՆՈՐ ՆՈՐՔ 8 ԶՆԳՎ.,նոր նորք 8 | զանգված,8-րդ,միկրոշրջան,1775
3,NaN,ՄԱԼԱԹԻԱ-ՍԵԲԱՍՏԻԱ,ԻՍԱԿՈՎԻ Փ.,իսակովի | փողոց,Ծովակալ Իսակովի,պողոտա,1708
4,NaN,ԱՐԱԲԿԻՐ,ԲԱՂՐԱՄՅԱՆ Փ.,բաղրամյան | փողոց,Մարշալ Բաղրամյան,պողոտա,1312
5,NaN,ԳՅՈՒՄՐԻ,Խ. ՀԱՅՐԻԿ ԽՃՂ.,խ հայրիկ | խճուղի,Խրիմյան Հայրիկի,փողոց,858
6,ԵՂՎԱՐԴ,ՆԱԻՐԻ,ԵՐԵՎԱՆՅԱՆ Փ.,երեվանյան | փողոց,Երևանյան,թաղամաս,625
7,ՋԵՐՄՈՒԿ,ՋԵՐՄՈՒԿ,ՁԱԽԱՓՆՅԱԿ Փ.,ձախափնյակ | փողոց,Ձախափնյակ,թաղամաս,577
8,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ՏԻԳՐԱՆ ՄԵԾԻ Փ.,տիգրան մեծի | փողոց,Տիգրան Մեծի,պողոտա,537
9,NaN,ՆՈՐ ՆՈՐՔ,ԳՅՈՒԼԻՔԵԽՎՅԱՆ ՆՐԲ.,գյուլիքեխվյան | նրբանցք,Հ. Գյուլիքևխյան,փողոց,396


## Step 5 · House, apartment, object type → cadastre id

House and apartment must be **identical** after cleaning; no SPR apartment means the cadastre row must have no building number either. If several ids remain, the SPR house type (`Տ`, `Շ`, `ՀՈՂ`) picks the matching object. A single id that is a different kind of object (garage for an apartment, land plot for a house) becomes a suggestion.

In [17]:
s05_match_addresses.run()

[05_match_addresses] rows with one street (found or suggested): 429,532
[05_match_addresses]    street suggested apartment_missing_in_spr                   240
[05_match_addresses]    street suggested apartment_not_found                      8,143
[05_match_addresses]    street suggested cadastre_duplicate                       1,146
[05_match_addresses]    street suggested house_has_no_units_in_cadastre             699
[05_match_addresses]    street suggested house_missing_in_spr                       811
[05_match_addresses]    street suggested house_not_found                         14,070
[05_match_addresses]    street suggested mapped                                  12,324
[05_match_addresses]    street suggested object_type_conflict                       518
[05_match_addresses]    street suggested several_objects                            116
[05_match_addresses]    street suggested street_text_number_conflicts_house         230
[05_match_addresses]    street found     apartme

In [18]:
q("""SELECT street_confirmed, address_status, address_pick, object_type_check, count(*) AS rows
     FROM spr_address_match GROUP BY ALL ORDER BY street_confirmed DESC, rows DESC""")

,street_confirmed,address_status,address_pick,object_type_check,rows
0,True,house_not_found,NaN,NaN,147974
1,True,apartment_not_found,NaN,NaN,82611
2,True,mapped,single,same,81475
3,True,cadastre_duplicate,NaN,NaN,23559
4,True,mapped,by_object_type,same,17611
5,True,house_has_no_units_in_cadastre,NaN,NaN,12392
6,True,mapped,single,unknown,8666
7,True,object_type_conflict,single,different,6582
8,True,street_text_number_conflicts_house,NaN,NaN,3805
9,True,apartment_missing_in_spr,NaN,NaN,2550


Street found, house missing in the cadastre — the houses that street *does* have:

In [19]:
ex = q("""SELECT a.spr_id, a.street_uid, a.house_k FROM spr_address_match a
          WHERE a.address_status = 'house_not_found' ORDER BY random() LIMIT 1""")
print(ex.to_string(index=False))
houses_on(int(ex.street_uid[0]))

 spr_id  street_uid house_k
 541871       29483      10


,cadastre_id,house,house_type,building,building_type
0,632509,10բ,շենք,1,բնակարան
1,632513,10բ,շենք,10,բնակարան
2,639032,10բ,շենք,11,բնակարան
3,632503,10բ,շենք,12,բնակարան
4,632504,10բ,շենք,13,բնակարան
...,...,...,...,...,...
159,635365,81,բնակելի տուն,NaN,NaN
160,1131641,81/4,բնակելի տուն,NaN,NaN
161,635367,82,բնակելի տուն,NaN,NaN
162,635368,82/1,բնակելի տուն,NaN,NaN


## Step 6 · Final status, issue categories, output files

`match_status`: mapped · suggested · ambiguous · not_found · out_of_scope.
Only rows found **in the place the SPR row names** are mapped; a hit in another Yerevan district, or in a wider search because SPR gives no district / settlement, is a suggestion.

In [20]:
s06_export.run()

[06_export]    not_found                 348,377
[06_export]    out_of_scope               97,980
[06_export]    mapped         medium      70,244
[06_export]    mapped         high        36,561
[06_export]    suggested                  32,517
[06_export]    ambiguous                  28,404
[06_export] written to /Users/nareabgaryan/Desktop/ISAAPR1/spr_cadastre_mapping/output


In [21]:
q("""SELECT match_status, match_confidence, count(*) AS rows,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
     FROM spr_mapping_result GROUP BY ALL ORDER BY rows DESC""")

,match_status,match_confidence,rows,pct
0,not_found,NaN,348377,56.73
1,out_of_scope,NaN,97980,15.96
2,mapped,medium,70244,11.44
3,mapped,high,36561,5.95
4,suggested,NaN,32517,5.30
5,ambiguous,NaN,28404,4.63


In [22]:
q("""SELECT r.issue_category, i.owner, i.match_status, count(*) AS rows, i.meaning
     FROM spr_mapping_result r JOIN issue_ref i ON i.code = r.issue_category
     GROUP BY ALL ORDER BY rows DESC""")

,issue_category,owner,match_status,rows,meaning
0,CAD_HOUSE_NOT_FOUND,CADASTRE,not_found,147974,"Street exists in the cadastre, this house number does not"
1,SPR_FOREIGN_ADDRESS,SPR,out_of_scope,96297,"Region is not one of the 11 cadastre regions (another country, ԼՂՀ..."
2,CAD_APARTMENT_NOT_FOUND,CADASTRE,not_found,82611,"House exists with apartments, this apartment number does not"
3,STREET_NOT_FOUND,REVIEW,not_found,66814,"No cadastre street with this name in the place (old name, typo bey..."
4,CAD_DUPLICATE_ADDRESS,CADASTRE,ambiguous,23559,Several cadastre ids with identical address and object type
5,STREET_TYPE_MISMATCH,REVIEW,not_found,14898,"Street name found, but SPR type (Փ., ՆՐԲ. ...) differs from the ca..."
6,SPR_STREET_TEXT_UNPARSED,SPR,not_found,14440,"Street text holds extra parts that change the address (2 streets, ..."
7,CAD_HOUSE_HAS_NO_UNITS,REVIEW,not_found,12392,SPR gives an apartment; the cadastre house has no apartments at all
8,ADDRESS_FOUND_ON_SUGGESTED_STREET,REVIEW,suggested,12324,"Street only suggested or type differs, but this exact house/apartm..."
9,STREET_STREETLESS_SUGGESTED,REVIEW,suggested,7563,SPR has no street; settlement has street-less cadastre addresses b...


In [23]:
q("""SELECT street_match_method, match_confidence, count(*) AS rows FROM spr_mapping_result
     WHERE match_status = 'mapped' GROUP BY ALL ORDER BY rows DESC""")

,street_match_method,match_confidence,rows
0,exact,high,32964
1,initial_for_name,medium,28571
2,extra_words,medium,23430
3,exact,medium,15814
4,word_form,high,2454
5,fuzzy,medium,1795
6,double_letter,high,982
7,word_form,medium,541
8,alias,high,161
9,double_letter,medium,90


## Step 7 · Integrity checks (the cell fails if any check fails)

In [24]:
s07_checks.run()

[07_checks] PASS  result has one row per SPR file row
[07_checks] PASS  spr_id is unique and runs 1..N
[07_checks] PASS  spr table equals the raw file (no value changed)
[07_checks] PASS  cadastre table equals the raw file
[07_checks] PASS  cadastre_id is set exactly for mapped rows
[07_checks] PASS  every mapped/suggested id exists in the cadastre
[07_checks] PASS  every non-mapped row has an issue category
[07_checks] PASS  mapped rows have a confidence, others do not
[07_checks] PASS  mapped: same region (raw values)
[07_checks] PASS  mapped: same house number (raw values)
[07_checks] PASS  mapped: same apartment / building number (raw values)
[07_checks] PASS  mapped Yerevan rows: same district (raw values)
[07_checks] info: 1,790 cadastre ids are the target of more than one SPR row (the same address written differently in SPR)


## Investigate

Pick any row and see every decision. Change the `spr_id`s, categories and texts below.

In [30]:
one = q("SELECT spr_id FROM spr_mapping_result WHERE match_status = 'mapped' AND street_match_method = 'extra_words' ORDER BY random() LIMIT 1").spr_id[0]
explain(int(one))

== SPR row 264987


,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,264987,None,None,ԵՐԵՎԱՆ,ԱՋԱՓՆՅԱԿ,None,ՖՈՒՉԻԿԻ 1 ՆՐԲ.,15,Շ,26



== Step 3 geography: rule=yerevan  status=ok  note=None  hint=None
   keys: region='երեվան' community='երեվան' settlement='աջափնյակ'
   search areas: [['settlement', ['երեվան', 'երեվան', 'աջափնյակ']], ['community', ['երեվան', 'երեվան']]]
   house_k='15' apartment_k='26' house_type='Շ'

== How the street text is read:
   as_written             ֆուչիկի 1 | նրբանցք                           leftover: none 
   lane_of_street         ֆուչիկի | 1 | նրբանցք                         leftover: none 

== Step 4 street: found  rule=extra_words  reading=lane_of_street  searched_in=settlement  type_check=issue has no type  score=nan


,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,16372,Աջափնյակ,Յու. Ֆուչիկի,փողոց,1-ին,նրբանցք,375



== Step 5 address: mapped  pick=single  object_type=same
   cadastre rows with this house on that street: 34


,cadastre_id,house,house_type,building,building_type
0,716520,15,շենք,40,բնակարան
1,716523,15,շենք,23,բնակարան
2,716524,15,շենք,24,բնակարան
3,716525,15,շենք,25,բնակարան
4,716526,15,շենք,26,բնակարան
5,716529,15,շենք,34,բնակարան
6,716530,15,շենք,35,բնակարան
7,716531,15,շենք,37,բնակարան
8,716533,15,շենք,39,բնակարան
9,716535,15,շենք,41,բնակարան



== Final


,cadastre_id,match_status,match_confidence,issue_category,issue_owner,issue_detail,suggested_cadastre_id
0,716526,mapped,medium,None,None,None,None


In [31]:
one = q("SELECT spr_id FROM spr_mapping_result WHERE issue_category = 'STREET_AMBIGUOUS' ORDER BY random() LIMIT 1").spr_id[0]
explain(int(one))

== SPR row 552714


,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,552714,None,None,ՇԻՐԱԿ,ԳՅՈՒՄՐԻ,None,ԽԱՉԱՏՐՅԱՆՆ Փ ՏՈՒՆ33,None,None,None



== Step 3 geography: rule=gyumri  status=ok  note=None  hint=None
   keys: region='շիրակ' community='գյումրի' settlement='գյումրի'
   search areas: [['settlement', ['շիրակ', 'գյումրի', 'գյումրի']]]
   house_k='' apartment_k='' house_type=''

== How the street text is read:
   as_written             խաչատրյանն | փողոց                            leftover: house տուն 33

== Step 4 street: ambiguous  rule=extra_words  reading=as_written  searched_in=settlement  type_check=same type  score=nan


,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,29473,Գյումրի,Անդրանիկ Խաչատրյան,փողոց,None,None,49
1,29491,Գյումրի,Արամ Խաչատրյան,փողոց,None,None,609



== Final


,cadastre_id,match_status,match_confidence,issue_category,issue_owner,issue_detail,suggested_cadastre_id
0,None,ambiguous,None,STREET_AMBIGUOUS,REVIEW,candidates (extra_words): Գյումրի: Անդրանիկ Խաչատրյան փողոց ; Գյու...,None


In [32]:
sample('OBJECT_TYPE_CONFLICT', 10)

,spr_id,region,community,settlement,street,house,house_type,apartment,cadastre_id,match_status,match_confidence,issue_category,issue_detail,street_match_method
0,336413,ԵՐԵՎԱՆ,ՄԱԼԱԹԻԱ-ՍԵԲԱՍՏԻԱ,NaN,ՍԵԲԱՍՏԻԱ Փ.,86,Շ,5,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 925056,word_form
1,34993,ԱՐՄԱՎԻՐ,ՄԵԾԱՄՈՐ,ՀԱՅԿԱՎԱՆ,15 Փ.,34,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 75015,exact
2,511957,ԿՈՏԱՅՔ,ՆԱԻՐԻ,ՔԱՍԱԽ,ԿԱՐԵՆ ԴԵՄԻՐՃՅԱՆ 1 Փ.,14,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 522331,initial_for_name
3,368730,ԵՐԵՎԱՆ,ՇԵՆԳԱՎԻԹ,NaN,ՆՈՐ ԽԱՐԲԵՐԴ 23 Փ.,62/1,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 1258220,extra_words
4,24405,ԿՈՏԱՅՔ,ԱԲՈՎՅԱՆ,ԱՌԻՆՋ,Ա ԹՂՄ. 3 Փ.,24,ՀՈՂ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type ՀՈՂ vs cadastre object of id 1224209,exact
5,79992,ԵՐԵՎԱՆ,ԿԵՆՏՐՈՆ,NaN,ԱՆՏԱՌԱՅԻՆ Փ.,168/1,Շ,7,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 856244,exact
6,483557,ԿՈՏԱՅՔ,ԱԲՈՎՅԱՆ,ԱԲՈՎՅԱՆ,8 ՄԿՐՇ. 2 ԹՂՄ.,24,ՀՈՂ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type ՀՈՂ vs cadastre object of id 446911,exact
7,83155,ԵՐԵՎԱՆ,ԿԵՆՏՐՈՆ,NaN,ՀԱՆՐԱՊԵՏՈՒԹՅԱՆ Փ.,78,Շ,48,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 911526,exact
8,368886,ԵՐԵՎԱՆ,ՇԵՆԳԱՎԻԹ,NaN,ՆՈՐ ԽԱՐԲԵՐԴ 26 Փ.,17,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 1258090,extra_words
9,86168,ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ԳՈԳՈԼԻ Փ.,24,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 979436,initial_for_name


In [33]:
search_cadastre('բաղրամյան', 'Արաբկիր')

,street_uid,region,community,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,16569,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,2-րդ,նրբանցք,46
1,16570,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,2-րդ,փակուղի,118
2,16571,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,3-րդ,նրբանցք,170
3,16572,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,NaN,NaN,1464


In [34]:
streets_in('Ծովազարդ').head(30)

,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,12364,Ծովազարդ,Ա. Բակունցի,փողոց,NaN,NaN,13
1,12365,Ծովազարդ,Ա. Իսահակյան,փողոց,1-ին,փակուղի,1
2,12366,Ծովազարդ,Ա. Իսահակյան,փողոց,2-րդ,փակուղի,1
3,12367,Ծովազարդ,Ա. Իսահակյան,փողոց,NaN,NaN,15
4,12368,Ծովազարդ,Աղբյուր Սերոբի,փողոց,NaN,NaN,15
5,12369,Ծովազարդ,Արաբոյի,փողոց,NaN,NaN,9
6,12370,Ծովազարդ,Գ. Նժդեհի,փողոց,NaN,NaN,6
7,12371,Ծովազարդ,Գ. Չաուշի,փողոց,NaN,NaN,22
8,12374,Ծովազարդ,Զորավար Անդրանիկի,փողոց,1-ին,նրբանցք,6
9,12375,Ծովազարդ,Զորավար Անդրանիկի,փողոց,1-ին,փակուղի,1


Output files are in `output/`:
`spr_mapping_result.csv` (every row), `00_summary.xlsx`, `issues_for_cadastre.xlsx`, `issues_for_spr.xlsx`, `needs_review.xlsx`, `spr_with_id.csv`.

In [39]:
import pandas as pd
df = pd.read_csv("output/spr_mapping_result.csv", dtype=str, encoding="utf-8-sig")
f = df[(df["street_match_method"] == "fuzzy")].copy()
f["similarity_score"] = f["similarity_score"].astype(float)   # so sorting is numeric
fuzzy_streets = (
   f.groupby([ "street","cad_street_name"],
             dropna=False)
    .agg(spr_rows=("spr_id", "size"), example_spr_id=("spr_id", "first"))
    .reset_index()
)
print(len(fuzzy_streets), "distinct street pairs")   # 134
fuzzy_streets.to_csv("output/unique_fuzzy_output.csv", index=False, encoding="utf-8-sig")


1075 distinct street pairs


In [26]:
fuzzy_mapped = q("""
SELECT # r.spr_id, r.region, r.community, r.settlement,
       r.street            AS spr_street,
      # r.house, r.apartment,
      c.street_name       AS cad_street
      # c.street_type       AS cad_type,
      # c.sub_street_name   AS cad_sub_street,
      # c.house             AS cad_house,
      # c.building          AS cad_building,
      # r.similarity_score,
      # r.text_reading,
      # r.cadastre_id
FROM spr_mapping_result r
JOIN cadastre c ON c.id = r.cadastre_id
WHERE r.match_status = 'mapped'
 AND r.street_match_method = 'fuzzy'
ORDER BY r.similarity_score, r.region, r.settlement, r.street
""")
print(len(fuzzy_mapped), "rows")
fuzzy_mapped.head(50)

ParserException: Parser Error: syntax error at or near "r"

LINE 2: SELECT # r.spr_id, r.region, r.community, r.settlement,
                 ^